# GitHub Actions Certification (GH-200) — Complete Tutorial & Exam Guide
## 85 Practice Questions with Detailed Explanations & Testable Workflows

This notebook covers **every question** from the GH-200 practice exam, organized by topic.
Each section includes:
- 📝 The exam question with answer
- 📖 Detailed explanation of the concept
- 💻 Working YAML workflow examples you can upload to GitHub to test

### Topic Areas
| # | Topic | Questions |
|---|---|---|
| 1 | Workflow Triggers & Events | Q7, Q9, Q14, Q19, Q20, Q24, Q31, Q43, Q51 |
| 2 | Environment Variables, Secrets & Outputs | Q3, Q10, Q13, Q15, Q16, Q18, Q21, Q23, Q26, Q27, Q28, Q30, Q35, Q37, Q42, Q49, Q62, Q75, Q77, Q80 |
| 3 | Runners (Self-Hosted & GitHub-Hosted) | Q4, Q5, Q6, Q22, Q69, Q70, Q78, Q82 |
| 4 | Workflow Commands & Debugging | Q1, Q2, Q41, Q55, Q72 |
| 5 | Conditional Logic, Matrix & Concurrency | Q11, Q17, Q34, Q36, Q38, Q44, Q45, Q47, Q48, Q50 |
| 6 | Custom Actions Development | Q8, Q12, Q25, Q46, Q52–Q58, Q60, Q61, Q64, Q66–Q68, Q73, Q74, Q76, Q81, Q85 |
| 7 | Artifacts, Caching & Storage | Q32, Q33, Q39, Q40, Q59, Q71 |
| 8 | Enterprise & Organization Policies | Q29, Q63, Q65, Q79, Q83, Q84 |

---

# Section 1: Workflow Triggers & Events

Understanding workflow triggers is fundamental to GitHub Actions. Workflows are triggered by
**events** — things that happen in or outside your repository.

---

### Q20: Where should workflow files be stored?

**Question:** Where should workflow files be stored to be triggered by events in a repository?

| Option | Answer |
|---|---|
| A) anywhere | ❌ |
| B) Nowhere; they must be attached to an action in the GitHub UI | ❌ |
| **C) `.github/workflows/`** | **✅ Correct** |
| D) `.workflows/` | ❌ |
| E) `.github/actions/` | ❌ |

> **Note:** The PDF marks B as correct, but this appears to be an error in the source material.
> The universally correct answer is **`.github/workflows/`** — this is where GitHub looks for
> workflow YAML files. Files must have `.yml` or `.yaml` extension.

---

### Q19: Manual Workflow Triggers

**Question:** Which workflow event is used to manually trigger a workflow run?

**✅ Answer: B) `workflow_dispatch`**

#### Explanation

`workflow_dispatch` allows you to manually trigger a workflow from the GitHub UI, API, or CLI.
You can define custom inputs that the user provides when triggering.

```yaml
on:
  workflow_dispatch:
    inputs:
      environment:
        description: 'Target environment'
        required: true
        default: 'staging'
        type: choice
        options:
          - staging
          - production
```

---

### Q7: Triggering Workflows from External Events

**Question:** You need to trigger a workflow using the GitHub API for activity that happens
outside of GitHub. Which workflow event do you use?

**✅ Answer: D) `repository_dispatch`**

#### Explanation

| Event | Purpose |
|---|---|
| `workflow_dispatch` | Manual trigger from GitHub UI/API (human-initiated) |
| `repository_dispatch` | Programmatic trigger from external systems via API |
| `workflow_run` | Trigger after another workflow completes |

```yaml
# Trigger via API:
# curl -X POST -H "Authorization: token $TOKEN" \
#   https://api.github.com/repos/OWNER/REPO/dispatches \
#   -d '{"event_type": "deploy", "client_payload": {"env": "prod"}}'

on:
  repository_dispatch:
    types: [deploy]
```

---

### Q14: Triggering After Another Workflow Completes

**Question:** You need to define a deployment workflow that runs after the build workflow
has successfully completed. Without modifying the build workflow, which trigger should you
define?

**✅ Answer: D) `workflow_run`**

#### Explanation

`workflow_run` lets you trigger a workflow based on the completion of another workflow.
It does NOT require modifying the upstream workflow.

```yaml
on:
  workflow_run:
    workflows: ["Build"]  # Name of the upstream workflow
    types:
      - completed

jobs:
  deploy:
    if: ${{ github.event.workflow_run.conclusion == 'success' }}
    runs-on: ubuntu-latest
    steps:
      - run: echo "Deploying after successful build"
```

---

### Q9: When Scheduled Workflows Run

**Question:** Scheduled workflows run on the:

**✅ Answer: E) Latest commit on the default or base branch**

Scheduled workflows always use the latest commit on the **default branch** (usually `main`).
They cannot be scheduled on feature branches.

---

### Q24: Cron Schedule Syntax

**Question:** Schedule a workflow twice a week at 7:45 UTC every Wednesday and Saturday.

**✅ Answer: C)**
```yaml
on:
  schedule:
    - cron: '45 7 * * 3,6'
```

#### Cron Syntax Reference
```
┌───────────── minute (0-59)
│ ┌───────────── hour (0-23)
│ │ ┌───────────── day of month (1-31)
│ │ │ ┌───────────── month (1-12)
│ │ │ │ ┌───────────── day of week (0-6, Sun=0)
│ │ │ │ │
45 7 * * 3,6   →  7:45 UTC on Wed(3) and Sat(6)
```

**Key syntax:** `on: schedule:` takes a list (`-`) of `cron:` entries.

---

### Q31: Multiple Event Triggers

**Question:** Which two code blocks allow a workflow to be triggered by multiple event types?

**✅ Answer: A and C**

```yaml
# Option A — Array syntax (simplest)
on: [push, pull_request]

# Option C — Detailed syntax with filters
on:
  schedule:
    - cron: '*/15 * * * *'
  # plus other events...
```

**Invalid events:** `commit` (not a valid event), `env:` (not an event)

---

### Q43: Pull Request Triggers with Filters

**Question:** Deploy based on PR labels, on the releases branch, only for changes in the apps folder.

**✅ Answer: C)**
```yaml
on:
  pull_request:
    types: [labeled]        # Trigger when label is added
    branches:
      - 'releases'          # Only the releases branch
    paths:
      - 'apps/**'           # Only changes under apps/
```

**Key distinctions:**
- `branches:` uses exact names or patterns (not `releases/**`)
- `paths:` uses glob patterns (`apps/**` not just `apps`)
- Event is `pull_request` with `types: [labeled]` (not `pull_request_label` or `pull_request_review`)

---

### Q51: Check Run Events

**Question:** Integrate with a third-party code quality provider using the Checks API.

**✅ Answer: B) Add the `check_run` webhook event when the code quality integration is completed**

```yaml
on:
  check_run:
    types: [completed]
```

In [ ]:
# Summary: Workflow Trigger Types

triggers = {
    'push': 'Code pushed to repository',
    'pull_request': 'PR opened/updated/labeled/etc.',
    'workflow_dispatch': 'Manual trigger (UI/API/CLI)',
    'repository_dispatch': 'External system trigger via API',
    'workflow_run': 'After another workflow completes',
    'schedule': 'Cron-based schedule (runs on default branch)',
    'check_run': 'When a check run completes (CI integrations)',
    'release': 'When a release is created/published',
}

print('GitHub Actions Trigger Types:')
print('=' * 65)
for trigger, desc in triggers.items():
    print(f'  {trigger:<25} {desc}')

print('\nKey Exam Points:')
print('  • Workflows live in .github/workflows/')
print('  • Scheduled workflows run on the default branch')
print('  • repository_dispatch = external triggers')
print('  • workflow_run = chain workflows without modifying upstream')
print('  • Cron syntax: minute hour day-of-month month day-of-week')

---

# Section 2: Environment Variables, Secrets & Outputs

---

### Q10: Environment Variable Scopes

**Question:** Custom environment variables can be defined at which levels? (Choose 3)

**✅ Answer: A) Top level, B) Step level, E) Job level**

```yaml
# TOP LEVEL — available to all jobs and steps
env:
  GLOBAL_VAR: "available-everywhere"

jobs:
  build:
    # JOB LEVEL — available to all steps in this job
    env:
      JOB_VAR: "available-in-this-job"
    runs-on: ubuntu-latest
    steps:
      - name: My step
        # STEP LEVEL — available only in this step
        env:
          STEP_VAR: "available-in-this-step"
        run: echo "$GLOBAL_VAR $JOB_VAR $STEP_VAR"
```

**Q18:** The **smallest scope** for an environment variable is a **step** (Answer: D).

**Q26:** The three scopes are: **entire workflow** (`env` at top), **job** (`jobs.<id>.env`),
and **step** (`jobs.<id>.steps[*].env`). (Answer: A, E, F)

---

### Q13: Environment Variable Syntax

**Question:** Proper syntax to specify a custom environment variable?

**✅ Answer: F) `env:` with `KEY: value` syntax**

```yaml
env:
  MY_VARIABLE: my-value    # ✅ Correct (colon, not equals)

# ❌ Wrong:
# var: MY_VARIABLE = my-value     (wrong keyword)
# environment: MY_VARIABLE = my-value  (wrong keyword)
# env: MY_VARIABLE = my-value     (equals sign, not colon)
```

---

### Q15: Step-Level Environment Variables

**Question:** Which step uses the dbserver environment variable correctly?

**✅ Answer: D)**
```yaml
steps:
  - name: Hello world
    run: echo $dbserver
    env:
      dbserver: orgserver1   # ✅ env: with key: value
```

**Wrong syntaxes:** `variables:`, `environment:`, `env: - name: ... value: ...` (list format)

---

### Q27: Setting Environment Variables for Subsequent Steps

**Question:** Which command sets `$FOO` for use in subsequent workflow steps?

**✅ Answer: B) `echo "FOO=bar" >> $GITHUB_ENV`**

```yaml
steps:
  - name: Set variable
    run: echo "FOO=bar" >> $GITHUB_ENV

  - name: Use variable
    run: echo "FOO is $FOO"   # Works! Variable persists across steps
```

**Why others are wrong:**
- `export FOO=bar` — Only lasts for the current shell/step
- `::set-env` — Deprecated for security reasons
- `${{ $FOO=bar }}` — Invalid expression syntax

---

### Q42: Setting Output Parameters

**Question:** Which command sets the output parameter for an action?

**✅ Answer: D) `echo "action_color=purple" >> $GITHUB_OUTPUT`**

```yaml
steps:
  - name: Set output
    id: my-step
    run: echo "action_color=purple" >> $GITHUB_OUTPUT

  - name: Use output
    run: echo "Color is ${{ steps.my-step.outputs.action_color }}"
```

**Key difference:**
- `$GITHUB_ENV` → environment variables (for current job)
- `$GITHUB_OUTPUT` → output parameters (can be used by downstream jobs via `needs`)

---

### Q49: Accessing Job Outputs from Dependent Jobs

**Question:** Syntax to access `output1` from `job1` in a dependent job?

**✅ Answer: A) `${{ needs.job1.outputs.output1 }}`**

```yaml
jobs:
  job1:
    runs-on: ubuntu-latest
    outputs:
      output1: ${{ steps.step1.outputs.result }}
    steps:
      - id: step1
        run: echo "result=hello" >> $GITHUB_OUTPUT

  job2:
    needs: job1
    runs-on: ubuntu-latest
    steps:
      - run: echo "${{ needs.job1.outputs.output1 }}"  # "hello"
```

---

### Q23 & Q30 & Q35 & Q37: Default Environment Variables

| Variable | Contains | Question |
|---|---|---|
| `GITHUB_REPOSITORY` | Owner and repo name (e.g., `octocat/Hello-World`) | Q23 ✅ |
| `GITHUB_REF` | Branch or tag that triggered the workflow | Q30 ✅ |
| `GITHUB_ACTOR` | Person or app that initiated the workflow | Q37 ✅ |
| `GITHUB_RUN_NUMBER` | Auto-incrementing run number | Q35 |

**Q35:** The proper syntax to reference `GITHUB_RUN_NUMBER`:
- In expressions: `${{ github.run_number }}`
- In shell: `$GITHUB_RUN_NUMBER`
- The PDF marks `${{GITHUB_RUN_NUMBER}}` as correct (which works as a shorthand)

---

### Q3 & Q16 & Q28: GITHUB_TOKEN

**Q3:** When should you use `GITHUB_TOKEN`?
**✅ A) When you want to make authenticated calls to the GitHub API**

**Q16:** Which scenarios could GITHUB_TOKEN be used for?
**✅ D) Publish to GitHub Packages, E) Create issues in the repo**

**Q28:** When must you explicitly use GITHUB_TOKEN?
**✅ A) Passing to an action that requires a token, B) Making authenticated API requests**

```yaml
steps:
  # Actions like checkout automatically use GITHUB_TOKEN
  - uses: actions/checkout@v4

  # Must explicitly pass for API calls or actions needing it
  - name: Create issue
    uses: some-action@v1
    with:
      token: ${{ secrets.GITHUB_TOKEN }}  # Explicit pass

  - name: API call
    run: |
      curl -H "Authorization: token ${{ secrets.GITHUB_TOKEN }}" \
        https://api.github.com/repos/${{ github.repository }}/issues
```

**GITHUB_TOKEN CANNOT:**
- ❌ Create repository secrets
- ❌ Add members to an organization
- ❌ Access other repositories (scoped to current repo)

---

### Q21, Q62, Q77, Q80: Secrets Management

**Q21:** Create a repo-specific version of an org secret → **Create a duplicate entry scoped to MyRepo**

**Q62:** Store a shared deployment token securely:
**✅ B) Corporate secret store (HashiCorp Vault), C) Organization-level encrypted secret**

**Q77:** Same reusable workflow for different cloud keys:
**✅ C) Store keys as environment secrets in different environments, pass via `secrets`**

**Q80:** Single secret for specific repositories:
**✅ B) Organization secret with "Selected repositories" access**

**Q75:** Valid scenario for environment secrets:
**✅ A) Configuring environment secrets for connecting to GitHub Enterprise Server**

In [ ]:
# Environment Variables & Secrets Quick Reference

print('=== Setting Values ===')
print('  $GITHUB_ENV    → Set env var for subsequent steps in same job')
print('  $GITHUB_OUTPUT → Set output parameter for downstream jobs')
print()
print('=== Accessing Values ===')
print('  ${{ env.MY_VAR }}              → Access env var in expressions')
print('  $MY_VAR                        → Access env var in shell')
print('  ${{ secrets.MY_SECRET }}       → Access secret')
print('  ${{ needs.job1.outputs.out1 }} → Access upstream job output')
print('  ${{ steps.step1.outputs.val }} → Access step output')
print()
print('=== Default Variables ===')
vars_table = {
    'GITHUB_REPOSITORY': 'owner/repo-name',
    'GITHUB_REF': 'refs/heads/branch-name',
    'GITHUB_ACTOR': 'username who triggered',
    'GITHUB_RUN_NUMBER': 'auto-incrementing run #',
    'GITHUB_SHA': 'commit SHA that triggered',
    'GITHUB_WORKSPACE': 'working directory on runner',
}
for var, desc in vars_table.items():
    print(f'  {var:<25} → {desc}')
print()
print('=== Secret Scopes ===')
print('  Repository secret    → Available to workflows in that repo')
print('  Environment secret   → Available to jobs using that environment')
print('  Organization secret  → Shared across selected/all repos in org')

---

# Section 3: Runners (Self-Hosted & GitHub-Hosted)

---

### Q4: When to Use Self-Hosted Runners

**Question:** In which scenarios should you use self-hosted runners? (Choose 2)

**✅ B) Jobs must run for longer than 6 hours**
**✅ C) Workflow job needs to install software from the local network**

GitHub-hosted runners have a **6-hour time limit** per job. Self-hosted runners have no such limit.
Self-hosted runners also provide access to your internal network resources.

**Why others are wrong:**
- GitHub-hosted runners already support Windows, macOS, and Linux
- GitHub Actions minutes are for GitHub-hosted runners, not self-hosted

---

### Q5: Service Containers

**Question:** Best way to use Redis on a self-hosted Linux runner without affecting future runs?

**✅ C) Specify `container:` and `services:` in your job definition**

```yaml
jobs:
  test:
    runs-on: ubuntu-latest
    services:
      redis:
        image: redis:7
        ports:
          - 6379:6379
        options: >-
          --health-cmd "redis-cli ping"
          --health-interval 10s
          --health-timeout 5s
          --health-retries 5
    steps:
      - run: redis-cli -h localhost ping
```

Service containers are **ephemeral** — they spin up for the job and are destroyed after,
leaving no impact on future workflow runs.

---

### Q6 & Q22: Custom Labels for Self-Hosted Runners

**Q6:** Prepare a self-hosted runner with custom software:
**✅ C) Inform users to identify the runner with labels `custom-software` and `linux`**
**✅ F) Add the label `custom-software` to the runner**

**Q22:** Target a specific self-hosted runner with XCode 11.2 on macOS:
**✅ C) Create custom labels `macos-10.15` and `xcode-11.2`**
**✅ E) `runs-on: [self-hosted, macos-10.15, xcode-11.2]`**
**✅ F) Assign the custom labels to the self-hosted runner**

```yaml
jobs:
  deploy:
    # All labels must match — this targets a specific runner
    runs-on: [self-hosted, macos-10.15, xcode-11.2]
    steps:
      - run: xcodebuild -version
```

---

### Q70: Targeting a Windows Self-Hosted Runner

**✅ Answer: A) `runs-on: [self-hosted, windows, x64]`**

Labels are comma-separated in an array. `self-hosted` is always included as the first label.

---

### Q78: GitHub-Hosted Runner Capabilities

**✅ A) Support for Linux, Windows, and macOS**
**✅ E) Automatic patching of both the runner and the underlying OS**

GitHub-hosted runners:
- ✅ Linux (Ubuntu), Windows, macOS
- ✅ Automatically patched and maintained
- ✅ Clean VM for every job (no state between jobs)
- ❌ No automatic file-system caching between runs
- ❌ No CentOS, Fedora, Debian variants

---

### Q69: IP Allow Lists and Runners

**✅ B) Use self-hosted runners with known IP addresses**
**✅ D) Use GitHub-hosted larger runners (configurable static IPs)**

Standard GitHub-hosted runners have dynamic IPs — they don't work with IP allow lists.

---

### Q82: Troubleshooting a Stalled Workflow

**Question:** Workflow specifying `runs-on: [gpu]` stalls until failing.

**✅ A) Review Runner_*.log files in the _diag folder**
**✅ D) Update org settings to enable GPU-based GitHub-hosted runners**

---

# Section 4: Workflow Commands & Debugging

---

### Q1: Printing Debug Messages

**Question:** How should you print a debug message in your workflow?

**✅ A) `echo "::debug::Set variable myVariable to true"`**

#### Workflow Command Syntax

```
echo "::COMMAND PARAMETER1=VALUE1::MESSAGE"
```

| Command | Example | Purpose |
|---|---|---|
| `debug` | `echo "::debug::My debug message"` | Debug log (only visible with debug logging enabled) |
| `warning` | `echo "::warning::This is a warning"` | Warning annotation |
| `error` | `echo "::error::This is an error"` | Error annotation |
| `notice` | `echo "::notice::FYI"` | Notice annotation |
| `add-mask` | `echo "::add-mask::secretvalue"` | Mask a value in logs |

---

### Q2: Workflow Commands Overview

**Question:** Which option lets you communicate with the runner machine to set env vars,
output values, add debug messages, etc.?

**✅ E) Workflow commands**

Workflow commands are the `::command::` syntax used in `run:` steps to interact with the runner.

---

### Q41: Enable Step Debug Logging

**Question:** How should you enable step debug logging?

**✅ D) Set the `ACTIONS_STEP_DEBUG` secret or variable at the repository level to `true`**

This is done in repo Settings → Secrets → New secret → `ACTIONS_STEP_DEBUG` = `true`.
You do NOT set it in the workflow file itself.

---

### Q55: Workflow Commands That Send Information

**Question:** Which workflow commands send information FROM the runner? (Choose 2)

**✅ B) Setting a debug message** (sends debug info to the log)
**✅ D) Setting output parameters** (sends output values to downstream steps/jobs)

---

### Q72: Error Message Syntax

**Question:** Which workflow commands set an error message? (Choose 2)

**✅ A) `echo "::error file=main.py,line=10,col=15::There was an error"`** (with file annotation)
**✅ C) `echo "::error::There was an error"`** (simple error)

Both formats are valid. The first adds file/line/column metadata for PR annotations.

In [ ]:
# Workflow Commands Cheat Sheet

commands = [
    ('Debug message',    'echo "::debug::My debug message"'),
    ('Warning',          'echo "::warning::This might be an issue"'),
    ('Error',            'echo "::error::Something went wrong"'),
    ('Error with file',  'echo "::error file=app.js,line=10::Bug here"'),
    ('Notice',           'echo "::notice::FYI information"'),
    ('Mask value',       'echo "::add-mask::my-secret-value"'),
    ('Set env var',      'echo "MY_VAR=value" >> $GITHUB_ENV'),
    ('Set output',       'echo "result=value" >> $GITHUB_OUTPUT'),
    ('Add to PATH',      'echo "/my/path" >> $GITHUB_PATH'),
    ('Group logs',       'echo "::group::My Group Title"'),
    ('End group',        'echo "::endgroup::"'),
]

print('GitHub Actions Workflow Commands:')
print('=' * 70)
for name, cmd in commands:
    print(f'  {name:<20} {cmd}')

print('\nDebugging:')
print('  ACTIONS_STEP_DEBUG=true  → Set as repo secret for verbose logs')
print('  ACTIONS_RUNNER_DEBUG=true → Runner-level debug logging')

---

# Section 5: Conditional Logic, Matrix, Environments & Concurrency

---

### Q11: Conditional Job Execution

**Question:** Execute deploy job only if current branch is `feature-branch`.

**✅ Answer: A)**
```yaml
jobs:
  deploy:
    if: github.ref_name == 'feature-branch'    # ✅ Correct context
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
```

**Wrong alternatives:** `github.branch.name`, `github.ref.name`, `github.branch_name` — these do not exist.

---

### Q17: Multi-Line Run Commands

**Question:** Run `npm ci` and `npm run build` as part of a workflow.

**✅ Answer: A) Use the pipe `|` for multi-line commands**
```yaml
- run: |
    npm ci
    npm run build
```

The `|` (literal block scalar) preserves newlines. Each line runs as a separate command.

**Wrong:** `shell:` is not a command runner, `shell: nodejs` doesn't exist.

---

### Q44: Matrix Strategy

**Question:** Build targeting multiple node versions.

**✅ Answer: A)**
```yaml
jobs:
  build:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        node-version: [14, 16, 18]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-node@v4
        with:
          node-version: ${{ matrix.node-version }}
      - run: npm ci && npm test
```

Matrix creates a job for **each combination** of values.

---

### Q36: Concurrency Control

**Question:** Mitigate risk of multiple deployments running simultaneously.

**✅ C) Specify a target environment in the deployment job**
**✅ D) Specify a concurrency scope in the workflow**

```yaml
# Method 1: Concurrency scope
concurrency:
  group: deploy-${{ github.ref }}    # Only one deploy per branch
  cancel-in-progress: true           # Cancel older runs

jobs:
  deploy:
    # Method 2: Environment (has built-in protection rules)
    environment: production
    runs-on: ubuntu-latest
    steps:
      - run: echo "Deploying..."
```

---

### Q38 & Q50: Environment Approvals

**Q38:** Require VMOps team approval before deployment:
**✅ B) Specify the environment in the workflow job**
**✅ C) Create a Production environment in REPO settings with required reviewers**

**Q50:** Ensure users approve before next step proceeds:
**✅ C) Add users as required reviewers for an environment**

```yaml
jobs:
  deploy:
    environment: production    # Links to environment protection rules
    runs-on: ubuntu-latest
    steps:
      - run: echo "This only runs after approval"
```

The environment and its reviewers are configured in **Repository Settings → Environments**
(not Organization settings).

---

### Q34: Workflow Approval Timeout

**Question:** How many days can a workflow be in the "Waiting" state before it fails?

**✅ A) 30 days**

---

### Q45: When to Disable a Workflow

**✅ B) Workflow error produces too many/wrong requests impacting external services**
**✅ D) Workflow sends requests to a service that is down**

Disabling lets you stop the workflow without deleting the file.

---

### Q47: Passing Data Between Jobs

**Question:** Two ways to pass data between jobs?

**✅ D) Artifact storage** (files)
**✅ E) Job outputs** (values)

---

### Q48: Caching vs Artifacts

**✅ B) Use caching when reusing files that change rarely between jobs/runs**
**✅ D) Use artifacts when referencing files produced by a job after a workflow ends**

| Feature | Caching | Artifacts |
|---|---|---|
| Purpose | Reuse dependencies (node_modules, pip) | Store build outputs, test results |
| Lifetime | ~7 days (LRU eviction) | 90 days default |
| Cross-run | ✅ Yes (that's the point) | ❌ Per-run |
| Downloadable | ❌ No | ✅ Yes (from UI/API) |

---

# Section 6: Custom Actions Development

---

### Action Types

| Type | `runs.using` | Best For | Files Required |
|---|---|---|---|
| **JavaScript** | `node16`/`node20` | Fast startup, cross-platform | `action.yml` + JS files |
| **Docker** | `docker` | Linux-only, full control over environment | `action.yml` + `Dockerfile` |
| **Composite** | `composite` | Bundle multiple run steps | `action.yml` |

### Q52: Simplest Action Type for Shell Scripts
**✅ C) JavaScript action** — Fastest to set up and run

### Q53: Best Action Type for TypeScript
**✅ D) JavaScript action** — TypeScript compiles to JavaScript

### Q73: Bundle Run Steps into Reusable Action
**✅ A) Composite action**

### Q68: Files Required for Docker Container Action
**✅ A) `action.yml` + D) `Dockerfile`**

---

### Q67: What File Defines Action Inputs and Outputs?

**✅ E) `action.yml`** — The metadata file defines everything about an action.

---

### Q58: How to Identify a JavaScript Action

**✅ D) The `action.yml` metadata file references a `package.json` file**

### Q60: How to Identify a Composite Action

**✅ D) `action.yml` has `runs.using` set to `composite`**

---

### Q54: Error Handling in JavaScript Actions

**Question:** Handle errors in a JavaScript action:
```javascript
const core = require('@actions/core');
try {
    // action code
} catch (error) {
    core.setFailed(error.message);   // ✅ Correct
}
```

**✅ C) `core.setFailed(error.message)`**

### Q61: Setting Failed Status from Code
**✅ D) A non-zero exit code** — Any script returning non-zero fails the step.

---

### Q57: Conditional Pre-Script in Action

**✅ Answer: B)**
```yaml
runs:
  using: 'node16'
  pre: 'start.js'                           # Script to run before main
  pre-if: github.event_name == 'push'       # Only on push events
  main: 'index.js'                          # Main entrypoint
```

---

### Q8: Where Can Actions Be Referenced?

**✅ B) Published Docker container image on Docker Hub**
**✅ C) Public NPM registry**
**✅ F) A separate public repository**

---

### Q12: Publishing Actions to GitHub Marketplace

**Mandatory requirements:**
**✅ B) Action's metadata file must be in the root directory of the repository**
**✅ C) Action's name cannot match a user/org on GitHub (unless that owner publishes it)**

---

### Q56: Referencing Action Versions

**Question:** Most common way to target a specific major release version?

**✅ A) `uses: actions/checkout@v3`** (major version tag)

### Q74: Best Practices for Publishing Actions

**✅ B) Commit SHA** (most secure, immutable)
**✅ D) Tag** (human-readable, practical)

```yaml
# By major version tag (most common)
- uses: actions/checkout@v4

# By commit SHA (most secure)
- uses: actions/checkout@ac593985615ec2ede58e132d2e21d2b1cbd6127c

# By branch (least recommended)
- uses: actions/checkout@main
```

---

### Q46: Reusable Workflow Reference

**✅ B) `uses: octo-org/another-repo/.github/workflows/workflow.yml@v1`**

Must include the full path: `{owner}/{repo}/.github/workflows/{file}@{ref}`

---

### Q25: Finding Actions for Unfamiliar Cloud Providers
**✅ B) Search GitHub Marketplace for verified actions published by the cloud provider**

### Q66: Testing a JavaScript Action
**✅ A) Create a workflow that only executes your specific JavaScript action**

### Q64: Running Python in a Step
```yaml
steps:
  - run: |
      import os
      print(os.environ['PATH'])
    shell: python    # ✅ Correct
```

### Q76: Reasons to Keep an Action in Its Own Repository
**✅ B) Easier for community to discover**
**✅ D) Decouples versioning from application code**

### Q81: Advantages of Action Documentation
**✅ B) Shares description of the action to users**
**✅ D) Creates a README for the consuming workflow**

### Q85: Best Practice in Enterprise Settings
**✅ A) Separate repository per action for independent version management**

---

# Section 7: Artifacts, Caching & Storage

---

### Q59: Cache Action Required Parameters

**✅ A) `path` — The file path on the runner to cache or restore**
**✅ C) `key` — The key created when saving/searching for a cache**

```yaml
- uses: actions/cache@v4
  with:
    path: ~/.npm                                    # Required: what to cache
    key: ${{ runner.os }}-npm-${{ hashFiles('**/package-lock.json') }}  # Required: cache key
    restore-keys: |                                 # Optional: fallback keys
      ${{ runner.os }}-npm-
```

---

### Q71: Publish to GitHub Container Registry

**Steps (Choose 3):**
**✅ A) Push the image**
**✅ B) Authenticate to the registry**
**✅ E) Build the container image**

```yaml
steps:
  - uses: actions/checkout@v4

  # Step 1: Authenticate
  - uses: docker/login-action@v3
    with:
      registry: ghcr.io
      username: ${{ github.actor }}
      password: ${{ secrets.GITHUB_TOKEN }}

  # Step 2: Build
  - run: docker build -t ghcr.io/${{ github.repository }}:latest .

  # Step 3: Push
  - run: docker push ghcr.io/${{ github.repository }}:latest
```

---

### Q32: Retrieve Workflow Run Logs via API

**✅ B) `GET /repos/:owner/:repo/actions/runs/:run_id/logs`**

---

### Q33: Preventing Storage Limit Issues

**✅ C) Configure artifact and log retention period**
**✅ E) Configure repo to use Git Large File Storage**

---

### Q39: Minimum Permission to Download Artifacts

**✅ E) Read** — The lowest permission needed.

### Q40: Deleting Workflow Runs

**✅ C) Completed workflow runs may be deleted** — Only completed (not pending/running).

---

# Section 8: Enterprise & Organization Policies

---

### Q29: Internal Repository Actions Access

**✅ B) Internal repos owned by the same organization**
**✅ D) Private repos owned by an organization of the enterprise**

---

### Q63: GitHub Actions on Enterprise Server

**✅ B) Third-party actions can be used via GitHub Connect**
**✅ C) Most GitHub-authored actions are automatically bundled**
**✅ F) Third-party actions can be manually synchronized**

---

### Q65: Trusted Actions in Enterprise Cloud

**✅ A) Actions can be published to an internal marketplace**
**✅ B) GitHub-verified actions can be collectively enabled**
**✅ C) Specific actions can be individually enabled (including version info)**

---

### Q79: Organization Policies — Local Actions Only

**✅ B) Distribute via repositories owned by the organization**

---

### Q83: Reusable Components Across Organization

**✅ D) Encrypted secrets** (organization-level)
**✅ E) Workflow templates** (starter workflows)
**✅ F) Actions stored in private repos in the organization**

---

### Q84: Preventing Certain Marketplace Actions

**✅ D) Create an allow-list of permitted actions as enterprise policy**

You restrict by **allowing only specific actions** — there is no "deny list" mechanism.

---

# 💻 Testable Workflow Files

The following workflows are ready to push to GitHub for testing.
They are saved in the `gh200_actions_tutorial/.github/workflows/` directory.

## How to Test

1. Create a new GitHub repository
2. Copy the `.github/workflows/` folder into your repo
3. Push to GitHub
4. Go to the **Actions** tab to see workflows run
5. For `workflow_dispatch` workflows, click **Run workflow** button

### Available Workflow Files:

| File | Tests Concepts |
|---|---|
| `01-triggers-and-events.yml` | Push, PR, schedule, manual dispatch |
| `02-env-vars-and-outputs.yml` | Env vars at all scopes, outputs, GITHUB_ENV |
| `03-matrix-and-conditions.yml` | Matrix strategy, conditional execution |
| `04-service-containers.yml` | Redis service container |
| `05-caching-and-artifacts.yml` | Dependency caching, artifact upload/download |
| `06-concurrency-and-environments.yml` | Concurrency groups, environment protection |
| `07-reusable-workflow.yml` | Reusable workflow with inputs |
| `07-caller-workflow.yml` | Workflow that calls the reusable workflow |
| `08-workflow-commands.yml` | Debug, warning, error, masking, grouping |
| `09-docker-build-push.yml` | Build and push to GHCR |
| `10-workflow-run-chain.yml` | Triggered after another workflow completes |

In [ ]:
# Quick reference: all 85 questions mapped to answers

answers = {
    1: 'A', 2: 'E', 3: 'A', 4: 'BC', 5: 'C', 6: 'CF', 7: 'D', 8: 'BCF',
    9: 'E', 10: 'ABE', 11: 'A', 12: 'BC', 13: 'F', 14: 'D', 15: 'D',
    16: 'DE', 17: 'A', 18: 'D', 19: 'B', 20: 'C (.github/workflows/)',
    21: 'B', 22: 'CEF', 23: 'A', 24: 'C', 25: 'B', 26: 'AEF', 27: 'B',
    28: 'AB', 29: 'BD', 30: 'A', 31: 'AC', 32: 'B', 33: 'CE', 34: 'A',
    35: 'D', 36: 'CD', 37: 'A', 38: 'BC', 39: 'E', 40: 'C', 41: 'D',
    42: 'D', 43: 'C', 44: 'A', 45: 'BD', 46: 'B', 47: 'DE', 48: 'BD',
    49: 'A', 50: 'C', 51: 'B', 52: 'C', 53: 'D', 54: 'C', 55: 'BD',
    56: 'A', 57: 'B', 58: 'D', 59: 'AC', 60: 'D', 61: 'D', 62: 'BC',
    63: 'BCF', 64: 'A', 65: 'ABC', 66: 'A', 67: 'E', 68: 'AD', 69: 'BD',
    70: 'A', 71: 'ABE', 72: 'AC', 73: 'A', 74: 'BD', 75: 'A', 76: 'BD',
    77: 'C', 78: 'AE', 79: 'B', 80: 'B', 81: 'BD', 82: 'AD', 83: 'DEF',
    84: 'D', 85: 'A'
}

print(f'GH-200 Answer Key ({len(answers)} questions)')
print('=' * 50)
for q in sorted(answers.keys()):
    print(f'  Q{q:>2}: {answers[q]}')

---

## 🎓 Exam Tips

1. **Workflow files** go in `.github/workflows/` (YAML format)
2. **`env:`** uses colon syntax (`KEY: value`), not equals
3. **`$GITHUB_ENV`** persists env vars across steps; **`$GITHUB_OUTPUT`** sets outputs
4. **`workflow_dispatch`** = manual, **`repository_dispatch`** = external API
5. **Self-hosted runners** for >6hr jobs, local network access, specific hardware
6. **Labels** target self-hosted runners: `runs-on: [self-hosted, label1, label2]`
7. **Matrix strategy** creates jobs for each combination of values
8. **Environment protection rules** are configured in repo settings, not org settings
9. **`action.yml`** is the metadata file that defines inputs, outputs, and runs config
10. **Composite actions** bundle multiple run steps; **JavaScript actions** for TypeScript/JS
11. **`core.setFailed()`** for JS action error handling; **non-zero exit code** for scripts
12. **Caching** = reuse across runs; **Artifacts** = download after run completes
13. **Organization secrets** with "Selected repositories" for cross-repo sharing
14. **Enterprise allow-list** to restrict which marketplace actions can run
15. **30 days** max waiting time for workflow approval